# Food-bank requirement prediction

This notebook trains a small PyTorch multilayer perceptron to predict `quantity_required` from food-bank, category, season, and distribution information.

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
FEATURE_COLUMNS = ["food_bank_id", "category", "season", "quantity_distributed"]
TARGET_COLUMN = "quantity_required"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
data_path = Path("../data/inventory_history.csv")
frame = pd.read_csv(data_path)

required_columns = set(FEATURE_COLUMNS + [TARGET_COLUMN])
missing_columns = required_columns.difference(frame.columns)
if missing_columns:
    raise ValueError(f"Missing required CSV columns: {sorted(missing_columns)}")

features = frame[FEATURE_COLUMNS].copy()
features["food_bank_id"] = features["food_bank_id"].astype(str)
categorical_columns = ["food_bank_id", "category", "season"]
category_values = {
    column: sorted(features[column].astype(str).unique().tolist())
    for column in categorical_columns
}
features = pd.get_dummies(
    features,
    columns=categorical_columns,
    dtype=np.float32,
)

targets = frame[TARGET_COLUMN].to_numpy(dtype=np.float32)
split_generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(features), generator=split_generator).numpy()
validation_size = max(1, int(len(features) * 0.2))
validation_indices = permutation[:validation_size]
training_indices = permutation[validation_size:]

distributed = features["quantity_distributed"]
training_mean = float(distributed.iloc[training_indices].mean())
training_std = float(distributed.iloc[training_indices].std()) or 1.0
features["quantity_distributed"] = (distributed - training_mean) / training_std

feature_tensor = torch.tensor(features.to_numpy(dtype=np.float32))
target_tensor = torch.tensor(targets)
training_dataset = TensorDataset(
    feature_tensor[training_indices], target_tensor[training_indices]
)
validation_dataset = TensorDataset(
    feature_tensor[validation_indices], target_tensor[validation_indices]
)
training_loader = DataLoader(training_dataset, batch_size=64, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=64)

print(f"Training records: {len(training_dataset)}")
print(f"Validation records: {len(validation_dataset)}")
print(f"Number of input features: {feature_tensor.shape[1]}")

In [ ]:
class RequirementPredictor(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, inputs):
        return self.network(inputs).squeeze(1)


model = RequirementPredictor(feature_tensor.shape[1])
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 100

for epoch in range(1, epochs + 1):
    model.train()
    for batch_features, batch_targets in training_loader:
        optimizer.zero_grad()
        predictions = model(batch_features)
        loss = loss_function(predictions, batch_targets)
        loss.backward()
        optimizer.step()

    model.eval()
    validation_loss = 0.0
    validation_count = 0
    with torch.no_grad():
        for batch_features, batch_targets in validation_loader:
            batch_loss = loss_function(model(batch_features), batch_targets)
            validation_loss += batch_loss.item() * len(batch_targets)
            validation_count += len(batch_targets)
    validation_loss /= validation_count

    if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
        print(f"Epoch {epoch:>3}/{epochs} - validation MSE: {validation_loss:.4f}")

In [ ]:
model.eval()
absolute_error = 0.0
sample_count = 0
with torch.no_grad():
    for batch_features, batch_targets in validation_loader:
        absolute_error += torch.abs(model(batch_features) - batch_targets).sum().item()
        sample_count += len(batch_targets)

validation_mae = absolute_error / sample_count
print(f"Validation MSE: {validation_loss:.4f}")
print(f"Validation MAE: {validation_mae:.4f}")

checkpoint = {
    "model_state_dict": model.state_dict(),
    "model_input_size": feature_tensor.shape[1],
    "feature_names": features.columns.tolist(),
    "category_values": category_values,
    "quantity_distributed_mean": training_mean,
    "quantity_distributed_std": training_std,
    "validation_mse": validation_loss,
    "validation_mae": validation_mae,
}
torch.save(checkpoint, "../quantity_required_model.pt")
print("Saved model checkpoint to ../quantity_required_model.pt")